# Phase 1: Scale-Up and Re-Labeling (USAspending CSV)

**Purpose:** Re-run the filtering and label construction pipeline on USAspending CSV downloads.

**Inputs:** `data/raw/usaspending/*.csv` (from zip download)
**Outputs:**
- `data/interim/filtered_physical_deliverables.csv`
- `data/processed/labeled_contracts.csv`
- Updated `data/processed/dataset_summary.txt`

**Key Design Decision:** PIID-group sampling preserves complete contract histories by selecting PIIDs first, then keeping all rows (modifications) for those contracts.

**Changes from Parquet version:**
- CSV files instead of Parquet shards
- Human-readable USAspending column names
- Pre-filtered on download (sanity checks only)


## 1. Environment Setup and Imports

In [1]:
# Data handling
import pandas as pd
import numpy as np
import zipfile
import os
import glob
import warnings
from pathlib import Path
from datetime import datetime

# Progress bar
from tqdm import tqdm

# Suppress noisy warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

# Global reproducibility seed
RANDOM_STATE = 42

print('Environment configured. Random state:', RANDOM_STATE)

Environment configured. Random state: 42


## 2. Configuration

All paths, filters, and thresholds defined as constants for reproducibility.

In [1]:
# --- File Paths ---
# Absolute path to shared data directory (parent of IS392 folder)
import os
from pathlib import Path

# Get the notebook directory and parent directory
NOTEBOOK_DIR = Path.cwd()
PARENT_DIR = Path(r'C:/Users/sarme/IS392Final/data')

# Default raw input folder for USAspending exports
RAW_INPUT_DIR = PARENT_DIR / 'raw' / 'usaspending'
# If the expected raw folder does not exist, fall back to PARENT_DIR (manual override friendly)
CSV_FOLDER = str(RAW_INPUT_DIR if RAW_INPUT_DIR.exists() else PARENT_DIR)
ZIP_FILE = None  # Set to path if CSVs are in a zip

# Intermediate checkpoint after filtering
INTERIM_OUTPUT = str(PARENT_DIR / 'interim' / 'filtered_physical_deliverables.csv')
# Final labeled dataset output
FINAL_OUTPUT = str(PARENT_DIR / 'processed' / 'labeled_contracts.csv')
# Dataset summary text file
SUMMARY_FILE = str(PARENT_DIR / 'processed' / 'dataset_summary.txt')

# Create output directories if they do not exist
os.makedirs(os.path.dirname(INTERIM_OUTPUT), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_OUTPUT), exist_ok=True)

# --- Filtering Criteria ---
# NOTE: Data is pre-filtered on USAspending download ($500K+, target PSC/NAICS)
# These filters act as sanity checks
PHYSICAL_PSC_PREFIXES = ['Y', 'Z']
PHYSICAL_PSC_NUMERIC_RANGE = (10, 99)
MIN_CONTRACT_VALUE = 500_000  # Sanity check - should already be applied

# --- Labeling Thresholds ---
# Primary cost overrun threshold: 5% growth triggers over_budget=1
COST_OVERRUN_THRESHOLD = 0.05
# Adaptive fallback threshold if positive class is too small (<5%)
COST_OVERRUN_THRESHOLD_FALLBACK = 0.01
# Any positive delay in days triggers late=1
SCHEDULE_DELAY_THRESHOLD = 0

# --- Sampling Configuration ---
# Target number of unique PIID contracts to sample (None = use all)
SAMPLE_CONTRACTS = None  # Set to integer (e.g., 50000) to sample, None to use all

# --- USAspending Column Mapping ---
# Maps semantic names to USAspending CSV column names
COLUMN_MAP = {
    'piid': 'award_id_piid',
    'mod_number': 'modification_number',
    'description': 'transaction_description',
    'psc': 'product_or_service_code',
    'naics': 'naics_code',
    'base_all_options': 'potential_total_value_of_award',
    'base_exercised': 'current_total_value_of_award',
    'obligated_amount': 'federal_action_obligation',
    'current_completion': 'period_of_performance_current_end_date',
    'ultimate_completion': 'period_of_performance_potential_end_date',
    'effective_date': 'period_of_performance_start_date',
    'signed_date': 'action_date',
    'contract_type': 'type_of_contract_pricing',
    'extent_competed': 'extent_competed',
    'num_offers': 'number_of_offers_received',
    'agency_id': 'awarding_agency_code',
    'vendor_name': 'recipient_name',
    # Note: state_code column may not be available in all USAspending downloads
    # 'state_code': 'place_of_performance_state',
    'mod_reason': 'action_type',
}

# List of columns to read (subset for efficiency)
COLUMNS_TO_READ = list(COLUMN_MAP.values())

print('Configuration loaded.')
print(f'  CSV folder: {CSV_FOLDER}')
print(f'  Columns mapped: {len(COLUMN_MAP)}')
print(f'  Sampling: {"All PIIDs" if SAMPLE_CONTRACTS is None else f"{SAMPLE_CONTRACTS:,} PIIDs"}')

Configuration loaded.
  CSV folder: C:\Users\sarme\IS392Final\data
  Columns mapped: 18
  Sampling: All PIIDs


## 3. Helper Functions

Reusable functions for filtering contracts and computing labels.

In [2]:
def is_physical_deliverable(psc_code: str) -> bool:
    """
    Check if a PSC code indicates a physical deliverable.
    
    Parameters
    ----------
    psc_code : str
        The product or service code to check.
    
    Returns
    -------
    bool
        True if PSC indicates construction (Y), maintenance (Z),
        or supplies/equipment (10-99 numeric range).
    """
    if pd.isna(psc_code):
        return False
    psc_str = str(psc_code).strip().upper()
    # Check for Y or Z prefix (construction or maintenance)
    if psc_str.startswith(('Y', 'Z')):
        return True
    # Check for numeric range 10-99 (supplies/equipment)
    try:
        psc_num = int(psc_str)
        return PHYSICAL_PSC_NUMERIC_RANGE[0] <= psc_num <= PHYSICAL_PSC_NUMERIC_RANGE[1]
    except (ValueError, IndexError):
        return False


def compute_cost_growth(base_val, final_val) -> float:
    """
    Compute percentage cost growth between base and final contract values.
    
    Parameters
    ----------
    base_val : numeric or str
        Initial contract value (potential_total_value_of_award).
    final_val : numeric or str
        Final contract value (current_total_value_of_award).
    
    Returns
    -------
    float
        Percentage growth as decimal (e.g., 0.05 = 5% growth).
        Returns np.nan if values are invalid or base is zero.
    """
    try:
        # Clean string values (remove $ and commas)
        base = float(str(base_val).replace('$', '').replace(',', '').strip())
        final = float(str(final_val).replace('$', '').replace(',', '').strip())
        if base == 0:
            return np.nan
        return (final - base) / abs(base)
    except (ValueError, TypeError):
        return np.nan


def compute_delay(current_date, ultimate_date) -> float:
    """
    Compute schedule delay in days between current and ultimate completion dates.
    
    Parameters
    ----------
    current_date : str or datetime
        Original/current completion date.
    ultimate_date : str or datetime
        Ultimate (final) completion date.
    
    Returns
    -------
    float
        Delay in days (positive = delayed, negative = early).
        Returns np.nan if dates are invalid.
    """
    try:
        current = pd.to_datetime(current_date)
        ultimate = pd.to_datetime(ultimate_date)
        return (ultimate - current).days
    except (ValueError, TypeError):
        return np.nan


print('Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay')

Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay


## 4. Data Loading and Schema Discovery

Load CSV files from zip or folder, concatenate into single DataFrame.

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

import pandas as pd

try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

# Fallbacks allow this cell to run even if previous config cells were skipped.
if 'CSV_FOLDER' not in globals():
    _default_parent = Path(r'C:/Users/sarme/IS392Final/data')
    _default_raw = _default_parent / 'raw' / 'usaspending'
    CSV_FOLDER = str(_default_raw if _default_raw.exists() else _default_parent)
    ZIP_FILE = None
    print(f'CSV_FOLDER was not defined; using fallback: {CSV_FOLDER}')

if 'ZIP_FILE' not in globals():
    ZIP_FILE = None

if 'COLUMNS_TO_READ' not in globals():
    COLUMNS_TO_READ = None
    print('COLUMNS_TO_READ was not defined; loading all columns when needed.')

if 'COLUMN_MAP' not in globals():
    COLUMN_MAP = {}

def load_usaspending_data(csv_folder, zip_file=None):
    """
    Load USAspending data from CSV or Parquet files from folder or zip file.
    
    Parameters
    ----------
    csv_folder : str
        Path to folder containing CSV or Parquet files
    zip_file : str or None
        Path to zip file containing CSV files (if applicable)
    
    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame from all data files
    """
    # Extract from zip if provided
    if zip_file and Path(zip_file).exists():
        print(f'Extracting from zip: {zip_file}')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(csv_folder)
        print(f'Extracted to: {csv_folder}')
    
    # Find all CSV and Parquet files
    csv_files = sorted(glob.glob(os.path.join(csv_folder, '*.csv')))
    parquet_files = sorted(glob.glob(os.path.join(csv_folder, '*.parquet')))

    # Prefer CSV exports when present; parquet is fallback only.
    if csv_files:
        all_files = csv_files
        if parquet_files:
            print(f'Found {len(csv_files)} CSV files and {len(parquet_files)} Parquet files')
            print('Using CSV files only (parquet ignored to avoid mixed-schema loads).')
    else:
        all_files = parquet_files
        if parquet_files:
            print(f'No CSV files found. Using {len(parquet_files)} Parquet files.')
    
    if not all_files:
        raise FileNotFoundError(
            f'No CSV or Parquet files found in {csv_folder}. '
            f'Verify CSV_FOLDER points to the correct location.'
        )

    if all_files and all_files[0].endswith('.parquet') and pq is None:
        raise ImportError(
            'Parquet files were selected but pyarrow is not installed. '
            'Install pyarrow or provide CSV files in CSV_FOLDER.'
        )
    
    # Load and concatenate all files
    dfs = []
    
    for i, file_path in enumerate(all_files, 1):
        filename = os.path.basename(file_path)
        print(f'  Loading {i}/{len(all_files)}: {filename}...', end=' ')
        try:
            if file_path.endswith('.csv'):
                # Read selected columns if available; otherwise load all columns.
                try:
                    if COLUMNS_TO_READ:
                        df = pd.read_csv(file_path, usecols=COLUMNS_TO_READ, low_memory=False)
                    else:
                        df = pd.read_csv(file_path, low_memory=False)
                except (ValueError, KeyError):
                    df = pd.read_csv(file_path, low_memory=False)
            elif file_path.endswith('.parquet'):
                # Read parquet file
                try:
                    if COLUMNS_TO_READ:
                        table = pq.read_table(file_path, columns=COLUMNS_TO_READ)
                    else:
                        table = pq.read_table(file_path)
                    df = table.to_pandas()
                except (ValueError, KeyError):
                    table = pq.read_table(file_path)
                    df = table.to_pandas()
            else:
                print('SKIPPED: unsupported extension')
                continue
            
            dfs.append(df)
            print(f'{len(df):,} rows')
        except Exception as e:
            print(f'ERROR: {e}')
            continue
    
    if not dfs:
        raise ValueError('No data could be loaded from available files')
    
    # Concatenate
    combined_df = pd.concat(dfs, ignore_index=True)
    
    print(f'\nTotal loaded: {len(combined_df):,} rows')
    print(f'Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
    
    return combined_df


# Load the data
df = load_usaspending_data(CSV_FOLDER, ZIP_FILE)

# Show column info
print(f'\nColumns in dataset: {len(df.columns)} columns')
if COLUMN_MAP:
    print('\nMissing mapped columns:')
    missing_count = 0
    for semantic, actual in COLUMN_MAP.items():
        if actual not in df.columns:
            print(f'  - {semantic}: {actual}')
            missing_count += 1
    if missing_count == 0:
        print('  All mapped columns found')
else:
    print('\nCOLUMN_MAP not defined in this kernel state; skipping mapped-column check.')

CSV_FOLDER was not defined; using fallback: C:\Users\sarme\IS392Final\data
COLUMNS_TO_READ was not defined; loading all columns when needed.
Found 1 CSV files and 589 Parquet files
  Loading 1/590: fpds_subsample.csv... 

C:\Users\sarme\AppData\Local\Temp\ipykernel_15348\3035069980.py:85: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


## 5. Filtering to Physical Deliverables

Sanity checks - data should already be pre-filtered on USAspending download.

In [5]:
# Ensure required imports/inputs exist even if cells were run out of order.
if 'pd' not in globals():
    import pandas as pd
if 'Path' not in globals():
    from pathlib import Path
if 'datetime' not in globals():
    from datetime import datetime

if 'df' not in globals():
    if 'load_usaspending_data' in globals() and 'CSV_FOLDER' in globals() and 'ZIP_FILE' in globals():
        print('df not found in kernel; loading data now...')
        df = load_usaspending_data(CSV_FOLDER, ZIP_FILE)
    else:
        raise NameError('df is not defined. Run Cell 9 (Data Loading and Schema Discovery) first.')

if 'COLUMN_MAP' not in globals() or 'psc' not in COLUMN_MAP:
    raise KeyError("COLUMN_MAP['psc'] is required. Run Cell 5 (Configuration) first.")

if 'INTERIM_OUTPUT' not in globals():
    _fallback_parent = Path(r'C:/Users/sarme/IS392Final/data')
    INTERIM_OUTPUT = str(_fallback_parent / 'interim' / 'filtered_physical_deliverables.csv')

if 'is_physical_deliverable' not in globals():
    def is_physical_deliverable(psc_code: str) -> bool:
        if pd.isna(psc_code):
            return False
        psc_str = str(psc_code).strip().upper()
        if psc_str.startswith(('Y', 'Z')):
            return True
        try:
            psc_num = int(psc_str)
            return 10 <= psc_num <= 99
        except (ValueError, TypeError):
            return False

# Collect statistics
run_metadata = {
    'run_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_rows_loaded': len(df),
    'total_rows_after_psc_check': 0,
    'total_rows_after_value_check': 0,
    'unique_piids_before_sample': 0,
    'unique_piids_after_sample': 0,
    'final_labeled_count': 0,
}

# Sanity check: PSC filter
print('Applying PSC sanity check...')
psc_col = COLUMN_MAP['psc']
if psc_col not in df.columns:
    raise KeyError(f"Required PSC column '{psc_col}' not found in loaded data.")

physical_mask = df[psc_col].apply(is_physical_deliverable)
df_filtered = df[physical_mask].copy()

psc_removed = len(df) - len(df_filtered)
psc_pct = (len(df_filtered) / len(df) * 100) if len(df) > 0 else 0

print(f'  Rows before: {len(df):,}')
print(f'  Rows after PSC check: {len(df_filtered):,} ({psc_pct:.1f}%)')
print(f'  Rows removed: {psc_removed:,}')

if psc_removed > len(df) * 0.1:
    print('\n  Warning: PSC filter removed >10% of rows. Check download filters.')

run_metadata['total_rows_after_psc_check'] = len(df_filtered)

# MOVED: Contract value filter now applied at PIID-group level (Step 7)
# This preserves complete contract histories for accurate baseline/final comparison
print('\nNote: Contract value filter will be applied at PIID-group level (Step 7)')
run_metadata['total_rows_after_value_check'] = len(df_filtered)

# Save filtered checkpoint
df_filtered.to_csv(INTERIM_OUTPUT, index=False)
print(f'\nSaved filtered checkpoint to: {INTERIM_OUTPUT}')

NameError: df is not defined. Run Cell 9 (Data Loading and Schema Discovery) first.

## 6. PIID-Group Sampling

Sample complete PIID groups to preserve contract modification histories. Set SAMPLE_CONTRACTS = None to use all data.

In [17]:
# Get unique PIIDs
piid_col = COLUMN_MAP['piid']
unique_piids = df_filtered[piid_col].unique()
total_unique_piids = len(unique_piids)

print(f'Total unique contracts (PIIDs): {total_unique_piids:,}')

run_metadata['unique_piids_before_sample'] = total_unique_piids

# Determine if sampling is needed
if SAMPLE_CONTRACTS is None or total_unique_piids <= SAMPLE_CONTRACTS:
    # Use all PIIDs
    sample_df = df_filtered.copy()
    actual_sample_size = total_unique_piids
    print(f'Using all {total_unique_piids:,} PIIDs (no sampling)')
else:
    # Sample PIIDs
    print(f'Target sample size: {SAMPLE_CONTRACTS:,}')
    rng = np.random.RandomState(RANDOM_STATE)
    sampled_piids = rng.choice(unique_piids, size=SAMPLE_CONTRACTS, replace=False)
    sample_df = df_filtered[df_filtered[piid_col].isin(sampled_piids)].copy()
    actual_sample_size = SAMPLE_CONTRACTS
    print(f'Sampled {SAMPLE_CONTRACTS:,} PIIDs -> {len(sample_df):,} total rows')

run_metadata['unique_piids_after_sample'] = actual_sample_size

# Reset index
sample_df = sample_df.reset_index(drop=True)
print(f'\nWorking sample shape: {sample_df.shape}')

Total unique contracts (PIIDs): 46,152
Using all 46,152 PIIDs (no sampling)

Working sample shape: (230088, 18)


## 7. Outcome Label Construction

Group by PIID to construct binary labels for cost overruns (`over_budget`) and schedule delays (`late`).

In [18]:
# Type-cast dollar columns
for col in [COLUMN_MAP['base_all_options'], COLUMN_MAP['base_exercised']]:
    sample_df[col] = pd.to_numeric(
        sample_df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )

# Type-cast date columns
for col in [COLUMN_MAP['current_completion'], COLUMN_MAP['ultimate_completion'],
            COLUMN_MAP['effective_date'], COLUMN_MAP['signed_date']]:
    sample_df[col] = pd.to_datetime(sample_df[col], errors='coerce')

# Sort by PIID and modification number
sample_df = sample_df.sort_values([piid_col, COLUMN_MAP['mod_number']])

print('Grouping by PIID to construct labels...')
label_rows = []
piids_filtered = 0

# Iterate through each PIID group
for piid, group in tqdm(sample_df.groupby(piid_col), total=sample_df[piid_col].nunique(), desc='Processing PIIDs'):
    # Get first (initial) and last (final) modification
    first = group.iloc[0]
    last = group.iloc[-1]
    
    # Extract cost values
    base_val = first[COLUMN_MAP['base_all_options']]
    final_val = last[COLUMN_MAP['base_exercised']]
    
    # PIID-group level value filter: skip if initial contract value < MIN_CONTRACT_VALUE
    if pd.notna(base_val) and base_val >= MIN_CONTRACT_VALUE:
        # Extract dates
        current_date = first[COLUMN_MAP['current_completion']]
        ultimate_date = last[COLUMN_MAP['ultimate_completion']]
        
        # Compute derived metrics
        cost_growth = compute_cost_growth(base_val, final_val)
        delay = compute_delay(current_date, ultimate_date)
        
        # Build label row
        row = {
            'piid': piid,
            'description': first[COLUMN_MAP['description']],
            'psc': first[COLUMN_MAP['psc']],
            'naics': first[COLUMN_MAP['naics']],
            'contract_type': first[COLUMN_MAP['contract_type']],
            'extent_competed': first[COLUMN_MAP['extent_competed']],
            'num_offers': first[COLUMN_MAP['num_offers']],
            'agency_id': first[COLUMN_MAP['agency_id']],
            'vendor_name': first[COLUMN_MAP['vendor_name']],
            'base_value': base_val,
            'final_value': final_val,
            'cost_growth_pct': cost_growth * 100 if pd.notna(cost_growth) else np.nan,
            'delay_days': delay,
            'num_modifications': len(group),
        }
        
        if 'state_code' in COLUMN_MAP:
            row['state_code'] = first[COLUMN_MAP['state_code']]
        
        label_rows.append(row)
    else:
        piids_filtered += 1

# Create labeled DataFrame
labeled_df = pd.DataFrame(label_rows)

print(f'\nLabeled DataFrame created: {len(labeled_df):,} rows')
print(f'  PIIDs filtered by value: {piids_filtered:,} (base_value < {MIN_CONTRACT_VALUE:,})')

Grouping by PIID to construct labels...


Processing PIIDs: 100%|██████████| 46152/46152 [00:13<00:00, 3421.21it/s]



Labeled DataFrame created: 45,456 rows
  PIIDs filtered by value: 696 (base_value < 500,000)


### 7.1 Apply Binary Labels and Adaptive Threshold

In [ ]:
# Apply initial cost overrun threshold (5%)
threshold = COST_OVERRUN_THRESHOLD
labeled_df['over_budget'] = (labeled_df['cost_growth_pct'] > threshold * 100).astype(int)

# Adaptive threshold: if positive class is too small (<5%), drop to 1%
positive_rate = labeled_df['over_budget'].mean()
if positive_rate < 0.05:
    threshold = COST_OVERRUN_THRESHOLD_FALLBACK
    labeled_df['over_budget'] = (labeled_df['cost_growth_pct'] > threshold * 100).astype(int)
    print(f'Adaptive threshold applied: {threshold:.0%} (original 5% yielded only {positive_rate:.2%} positives)')
else:
    print(f'Using standard threshold: {threshold:.0%}')

# Schedule delay label (any positive delay = late)
labeled_df['late'] = (labeled_df['delay_days'] > SCHEDULE_DELAY_THRESHOLD).astype(int)

# Drop rows with missing labels
before_drop = len(labeled_df)
labeled_df = labeled_df.dropna(subset=['cost_growth_pct', 'delay_days']).copy()
after_drop = len(labeled_df)
valid_pct = (after_drop / before_drop * 100) if before_drop > 0 else 0

print(f'\nValid labels: {after_drop:,} / {before_drop:,} ({valid_pct:.1f}%)')

# Print class balance
print('\nClass Balance Summary:')
print('-' * 40)
total = len(labeled_df)
if total == 0:
    print('  No labeled rows available after filtering.')
else:
    for target in ['over_budget', 'late']:
        pos = labeled_df[target].sum()
        print(f'  {target:12s}: {pos:6,} positive ({pos/total*100:5.2f}%) | {total-pos:6,} negative ({(total-pos)/total*100:5.2f}%)')

run_metadata['final_labeled_count'] = after_drop
run_metadata['over_budget_positives'] = int(labeled_df['over_budget'].sum())
run_metadata['late_positives'] = int(labeled_df['late'].sum())

Using standard threshold: 5%

Valid labels: 45,456 / 45,456 (100.0%)

Class Balance Summary:
----------------------------------------
  over_budget :  4,725 positive (10.39%) | 40,731 negative (89.61%)
  late        : 27,961 positive (61.51%) | 17,495 negative (38.49%)


### 7.2 Additional Derived Features

In [20]:
# Log-transformed initial cost
labeled_df['log_initial_cost'] = np.log1p(labeled_df['base_value'].fillna(0))

# Clean num_offers
labeled_df['num_offers'] = pd.to_numeric(labeled_df['num_offers'], errors='coerce').fillna(0)

# Value tier bins
labeled_df['value_tier'] = pd.cut(
    labeled_df['base_value'],
    bins=[0, 1e6, 5e6, 25e6, 100e6, float('inf')],
    labels=['<1M', '1M-5M', '5M-25M', '25M-100M', '>100M']
)

print('Derived features created:')
print('  - log_initial_cost')
print('  - num_offers (cleaned numeric)')
print('  - value_tier (categorical bins)')

Derived features created:
  - log_initial_cost
  - num_offers (cleaned numeric)
  - value_tier (categorical bins)


## 8. Save Outputs and Generate Summary

In [21]:
# Save final labeled dataset
labeled_df.to_csv(FINAL_OUTPUT, index=False)
print(f'Saved {len(labeled_df):,} labeled contracts to:')
print(f'   {FINAL_OUTPUT}')

Saved 45,456 labeled contracts to:
   ../data/processed/labeled_contracts.csv


### 8.1 Generate Dataset Summary Report

In [ ]:
# Build summary report
def _pct(numerator, denominator):
    return (numerator / denominator * 100) if denominator else 0.0

# Fallbacks for out-of-order execution
if 'run_metadata' not in globals():
    run_metadata = {}
if 'SUMMARY_FILE' not in globals() or 'INTERIM_OUTPUT' not in globals() or 'FINAL_OUTPUT' not in globals():
    from pathlib import Path
    _fallback_parent = Path(r'C:/Users/sarme/IS392Final/data')
    INTERIM_OUTPUT = globals().get('INTERIM_OUTPUT', str(_fallback_parent / 'interim' / 'filtered_physical_deliverables.csv'))
    FINAL_OUTPUT = globals().get('FINAL_OUTPUT', str(_fallback_parent / 'processed' / 'labeled_contracts.csv'))
    SUMMARY_FILE = globals().get('SUMMARY_FILE', str(_fallback_parent / 'processed' / 'dataset_summary.txt'))

total_loaded = run_metadata.get('total_rows_loaded', 0)
psc_kept = run_metadata.get('total_rows_after_psc_check', 0)
value_kept = run_metadata.get('total_rows_after_value_check', 0)
final_count = run_metadata.get('final_labeled_count', 0)
over_pos = run_metadata.get('over_budget_positives', 0)
late_pos = run_metadata.get('late_positives', 0)
run_date = run_metadata.get('run_date', 'N/A')

summary_lines = [
    '=' * 70,
    'DATASET SUMMARY REPORT - USAspending CSV',
    '=' * 70,
    f"Run Date: {run_date}",
    '',
    '-' * 70,
    'DATA LOADING',
    '-' * 70,
    f"Total rows loaded: {total_loaded:,}",
    '',
    '-' * 70,
    'FILTERING SANITY CHECKS',
    '-' * 70,
    f"After PSC check: {psc_kept:,}",
    f"After value check: {value_kept:,}",
    f"PSC retention rate: {_pct(psc_kept, total_loaded):.2f}%",
    '',
    '-' * 70,
    'SAMPLING',
    '-' * 70,
    f"Unique PIIDs before sampling: {run_metadata.get('unique_piids_before_sample', 0):,}",
    f"Sampled PIIDs: {run_metadata.get('unique_piids_after_sample', 0):,}",
    '',
    '-' * 70,
    'LABEL DISTRIBUTION',
    '-' * 70,
    f"Final labeled contracts: {final_count:,}",
    '',
    f"over_budget: {over_pos:,} positive ({_pct(over_pos, final_count):.2f}%)",
    f"             {final_count - over_pos:,} negative ({_pct(final_count - over_pos, final_count):.2f}%)",
    '',
    f"late:        {late_pos:,} positive ({_pct(late_pos, final_count):.2f}%)",
    f"             {final_count - late_pos:,} negative ({_pct(final_count - late_pos, final_count):.2f}%)",
    '',
    '-' * 70,
    'OUTPUT FILES',
    '-' * 70,
    f"Intermediate: {INTERIM_OUTPUT}",
    f"Final:        {FINAL_OUTPUT}",
    '=' * 70,
]

# Write summary
summary_text = '\n'.join(summary_lines)
with open(SUMMARY_FILE, 'w', encoding='utf-8') as f:
    f.write(summary_text)

# Print summary
print(summary_text)
print(f'\nSummary saved to: {SUMMARY_FILE}')

DATASET SUMMARY REPORT - USAspending CSV
Run Date: 2026-04-24 14:18:44

----------------------------------------------------------------------
DATA LOADING
----------------------------------------------------------------------
Total rows loaded: 233,642

----------------------------------------------------------------------
FILTERING SANITY CHECKS
----------------------------------------------------------------------
After PSC check: 230,088
After value check: 230,088
PSC retention rate: 98.48%

----------------------------------------------------------------------
SAMPLING
----------------------------------------------------------------------
Unique PIIDs before sampling: 46,152
Sampled PIIDs: 46,152

----------------------------------------------------------------------
LABEL DISTRIBUTION
----------------------------------------------------------------------
Final labeled contracts: 45,456

over_budget: 4,725 positive (10.39%)
             40,731 negative (89.61%)

late:        27,96

## 9. Next Steps

This notebook produces the foundation dataset for all downstream analysis. Next:

1. **Run `05_text_preprocessing_lda.ipynb`** — Apply two-track text preprocessing
2. **Run `06_classification.ipynb`** — Build feature matrices, train classification models
3. **(Optional) Run `07_gao_validation.ipynb`** — Cross-validate with GAO data

**Key outputs:**
- `data/processed/labeled_contracts.csv` — Complete labeled dataset
- `data/processed/dataset_summary.txt` — Metadata and class balance summary